# 💳 End-to-End Credit Card Fraud Detection Pipeline

Welcome to the comprehensive walkthrough of the **Credit Card Fraud Detection** system.

### Project Highlights:
1. **Data Understanding & Cleaning**: Ingesting 284,807 transactions, handling extreme class imbalance (~0.17%), and deduplicating records.
2. **Exploratory Data Analysis (EDA)**: Dissecting transaction amounts, temporal diurnal patterns, and PCA feature correlations.
3. **SQL Analytics**: Executing production queries for executive KPIs, risk tiering, and outlier surveillance.
4. **Imbalanced Machine Learning**: Comparing Baseline Logistic Regression against **SMOTE-resampled** Logistic Regression and **Random Forest**.
5. **Model Evaluation & Selection**: Multi-metric comparison (**PR-AUC**, **Recall**, **Precision**, **F1-Score**, **ROC-AUC**) prioritizing financial fraud prevention.
6. **Power BI Data Exports**: Aggregating analytical tables for interactive dashboards.

## 1. Environment Setup & Data Ingestion

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_data, inspect_dataset, clean_dataset
from src.eda import add_derived_features

df_raw = load_raw_data(PROJECT_ROOT / "data" / "raw" / "creditcard.csv")
summary = inspect_dataset(df_raw)
df_cleaned, cleaning_meta = clean_dataset(df_raw, drop_duplicates=True)

print(f"Raw dataset: {df_raw.shape[0]:,} rows | Cleaned dataset: {df_cleaned.shape[0]:,} rows")
print(f"Removed {cleaning_meta['removed_duplicates']:,} duplicates ({cleaning_meta['final_fraud']} fraud cases retained)")

## 2. Exploratory Data Analysis (EDA)

Let's explore class distribution, transaction amounts across classes, amount tiers, and hourly fraud rates.

In [ ]:
df_feat = add_derived_features(df_cleaned)

# Amount Statistics by Class
print("=== TRANSACTION AMOUNT SUMMARY (€ EUR) ===")
print(df_feat.groupby('Class')['Amount'].describe().round(2))

# Amount Category Fraud Rates
print("\n=== AMOUNT CATEGORY FRAUD RATES ===")
cat_summary = df_feat.groupby('Amount_Category', observed=False).agg(
    Total_Txns=('Class', 'count'),
    Fraud_Txns=('Class', 'sum'),
    Fraud_Rate_Pct=('Class', lambda x: (x.sum() / len(x)) * 100)
).round(4)
print(cat_summary)

### Display Exported EDA Visualizations

In [ ]:
from IPython.display import Image, display

images = [
    PROJECT_ROOT / "images" / "01_class_distribution.png",
    PROJECT_ROOT / "images" / "02_fraud_vs_legit_amount.png",
    PROJECT_ROOT / "images" / "03_amount_category_fraud_rate.png",
    PROJECT_ROOT / "images" / "04_hourly_fraud_trend.png",
    PROJECT_ROOT / "images" / "05_correlation_matrix.png"
]

for img_path in images:
    if img_path.exists():
        display(Image(filename=str(img_path)))

## 3. SQL Analytics Validation

Let's run our SQL queries in-memory using SQLite to verify analytical aggregations.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
df_cleaned.to_sql('creditcard_transactions', conn, index=False)

# Query: Amount Category Fraud Rates
query = """
WITH categorized AS (
    SELECT 
        Amount,
        Class,
        CASE 
            WHEN Amount <= 20.00 THEN '1. Low (€0 - €20)'
            WHEN Amount > 20.00 AND Amount <= 100.00 THEN '2. Medium (€20 - €100)'
            WHEN Amount > 100.00 AND Amount <= 500.00 THEN '3. High (€100 - €500)'
            ELSE '4. Very High (€500+)'
        END AS amount_tier
    FROM creditcard_transactions
)
SELECT 
    amount_tier,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS fraud_transactions,
    ROUND(CAST(SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS FLOAT) / COUNT(*) * 100.0, 4) AS fraud_rate_pct,
    ROUND(SUM(Amount), 2) AS total_amount_eur
FROM categorized
GROUP BY amount_tier
ORDER BY amount_tier ASC;
"""

pd.read_sql_query(query, conn)

## 4. Machine Learning & Resampling (SMOTE)

We evaluate **Baseline Logistic Regression**, **SMOTE Logistic Regression**, and **SMOTE Random Forest** on a stratified 20% test partition.

In [ ]:
comp_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "model_comparison.csv")
display(comp_df)

display(Image(filename=str(PROJECT_ROOT / "images" / "06_roc_pr_curves.png")))
display(Image(filename=str(PROJECT_ROOT / "images" / "07_confusion_matrices.png")))

## 5. Champion Model Metadata & Inference Demo

Let's load our serialized champion model and run inference on sample transactions.

In [ ]:
import json
import joblib

# Load serialized assets
model = joblib.load(PROJECT_ROOT / "models" / "best_fraud_model.joblib")
scaler = joblib.load(PROJECT_ROOT / "models" / "scaler.joblib")

with open(PROJECT_ROOT / "models" / "model_metadata.json", 'r') as f:
    metadata = json.load(f)

print("Champion Model:", metadata["model_name"])
print("PR-AUC Score:", metadata["metrics_on_test_set"]["pr_auc"])
print("Recall:", metadata["metrics_on_test_set"]["recall"])
print("Precision:", metadata["metrics_on_test_set"]["precision"])

# Top Feature Importances
top_feats = pd.DataFrame(list(metadata["top_10_features_by_importance"].items()), columns=["Feature", "Importance"])
print("\nTop 10 Feature Importances:")
print(top_feats.to_string(index=False))

## 6. Power BI Dashboard Exports Verification

Let's inspect the 4 exported dashboard tables in `data/processed/dashboard/`.

In [1]:
dashboard_files = list((PROJECT_ROOT / "data" / "processed" / "dashboard").glob("*.csv"))
for f in dashboard_files:
    df_dash = pd.read_csv(f)
    print(f"\n📁 {f.name} ({len(df_dash)} rows):")
    print(df_dash.head(3).to_string(index=False))

NameError: name 'PROJECT_ROOT' is not defined